# S3 — 分布偵探：Distribution & Outlier Analysis（Solution）

> **時間**：2 小時（講授 70min + 課堂練習 40min + QA 10min）  
> **資料**：`../datasets/titanic/train.csv`（類別型）、`../datasets/house-prices/train.csv`（數值型）  
> **學完能幹嘛**：看穿每個欄位的分布特性，找出離群值，判斷哪些特徵可能對預測有用

---

**一句話口訣：每個欄位都有自己的故事 — 偏態告訴你資料的脾氣，離群值告訴你資料的秘密。**

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

df = pd.read_csv('../datasets/titanic/train.csv')
hp = pd.read_csv('../datasets/house-prices/train.csv')

---
## 核心觀念 1／3 — 數值型分布三板斧

In [ ]:
hp.select_dtypes(include='number').hist(figsize=(16, 12), bins=30, edgecolor='white')
plt.suptitle('House Prices — 所有數值欄位分布', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def plot_kde_by_target(df, feature, target, title=None, figsize=(8, 5)):
    fig, ax = plt.subplots(figsize=figsize)
    for val in sorted(df[target].dropna().unique()):
        subset = df[df[target] == val][feature].dropna()
        sns.kdeplot(subset, fill=True, ax=ax, label=f'{target}={val}')
    ax.set_title(title or f'{feature} by {target}', fontsize=14)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_kde_by_target(df, 'Age', 'Survived', 'Age 分布：存活 vs 死亡')
plot_kde_by_target(df, 'Fare', 'Survived', 'Fare 分布：存活 vs 死亡')

In [ ]:
# log 轉換 SalePrice
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(hp['SalePrice'], bins=30, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('SalePrice 原始分布')
sns.histplot(np.log1p(hp['SalePrice']), bins=30, kde=True, ax=axes[1], color='coral')
axes[1].set_title('SalePrice log 轉換後')
plt.tight_layout()
plt.show()

---
## 核心觀念 2／3 — 類別型交叉比較

In [ ]:
def bar_chart_stacked(df, feature, target='Survived', stacked=True):
    survived = df[df[target] == 1][feature].value_counts()
    dead = df[df[target] == 0][feature].value_counts()
    compare = pd.DataFrame({'Survived': survived, 'Dead': dead})
    ax = compare.plot(kind='bar', stacked=stacked, figsize=(8, 5),
                      color=['#66b2ff', '#ff9999'], edgecolor='white')
    ax.set_title(f'{feature} vs {target}', fontsize=14)
    plt.tight_layout()
    plt.show()

bar_chart_stacked(df, 'Sex')
bar_chart_stacked(df, 'Pclass')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x='Sex', y='Survived', hue='Pclass', data=df, ax=ax)
ax.set_title('存活率 by Sex × Pclass', fontsize=14)
plt.tight_layout()
plt.show()

print(pd.crosstab(df['Sex'], df['Survived'], normalize='index').round(3))

In [ ]:
def plot_swarm(df, x, y, hue='Survived', title=None):
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.swarmplot(data=df, x=x, y=y, hue=hue, ax=ax, size=3)
    ax.set_title(title or f'{y} by {x}', fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_swarm(df, 'Sex', 'Age', title='Age × Sex × Survived')

---
## 核心觀念 3／3 — 離群值偵測

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(hp['GrLivArea'], hp['SalePrice'], alpha=0.5, s=20)
outliers = hp[hp['GrLivArea'] > 4000]
ax.scatter(outliers['GrLivArea'], outliers['SalePrice'],
           color='red', s=100, zorder=5, label=f'離群值 ({len(outliers)} 筆)')
ax.set_xlabel('GrLivArea')
ax.set_ylabel('SalePrice')
ax.set_title('GrLivArea vs SalePrice', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 分箱技巧
categories = ['cheap', 'standard', 'expensive', 'luxury']
df['Fare_bin'] = pd.qcut(df['Fare'], q=4, labels=categories)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x='Fare_bin', y='Survived', data=df, ax=ax, order=categories)
ax.set_title('存活率 by 票價分組', fontsize=14)
plt.tight_layout()
plt.show()

---
## 課堂練習 — Solution

### 🟢 送分題

In [ ]:
# 🟢 送分題 Solution
hp.select_dtypes(include='number').hist(figsize=(16, 12), bins=30, edgecolor='white')
plt.suptitle('House Prices — 所有數值欄位 Histogram', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### 🟡 核心題

In [ ]:
# 🟡 核心題 Solution

# 1. 偏態前 3 名
skew_top3 = hp.select_dtypes(include='number').skew().sort_values(ascending=False).head(3)
print('偏態前 3 名：')
print(skew_top3)

# 2-3. log 轉換 + 前後 KDE 比較
top3_cols = skew_top3.index.tolist()

fig, axes = plt.subplots(3, 2, figsize=(12, 12))
for i, col in enumerate(top3_cols):
    data = hp[col].dropna()
    data_pos = data[data > 0]  # log 需要正數
    
    sns.kdeplot(data, fill=True, ax=axes[i, 0], color='steelblue')
    axes[i, 0].set_title(f'{col} 原始 (skew={data.skew():.2f})', fontsize=11)
    
    log_data = np.log1p(data_pos)
    sns.kdeplot(log_data, fill=True, ax=axes[i, 1], color='coral')
    axes[i, 1].set_title(f'{col} log 轉換 (skew={log_data.skew():.2f})', fontsize=11)

plt.tight_layout()
plt.show()

### 🔴 挑戰題

In [ ]:
# 🔴 挑戰題 Solution
df_ex = df.copy()

# 1. 分箱
df_ex['Age_bin'] = pd.cut(
    df_ex['Age'], bins=[0, 12, 18, 35, 60, 80],
    labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Senior']
)

# 2. 各年齡組存活率
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x='Age_bin', y='Survived', data=df_ex, ax=ax)
ax.set_title('存活率 by 年齡組', fontsize=14)
ax.set_ylabel('存活率')
plt.tight_layout()
plt.show()

# 3. 加上 Sex
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x='Age_bin', y='Survived', hue='Sex', data=df_ex, ax=ax)
ax.set_title('存活率 by Age_bin × Sex', fontsize=14)
ax.set_ylabel('存活率')
plt.tight_layout()
plt.show()

# 4. 發現
print('觀察發現：')
print('1. Child（< 12 歲）不分性別，存活率都是最高的 — 兒童優先原則')
print('2. 女性在所有年齡組都比男性存活率高')
print('3. 男性 Senior（> 60 歲）存活率特別低')

---
## 小結與速查表

| 想做什麼 | 怎麼寫 |
|:---------|:-------|
| 全部 histogram | `df.select_dtypes(include='number').hist()` |
| KDE 疊圖 | `sns.kdeplot(subset, fill=True, label=...)` |
| 偏態值 | `df.skew().sort_values(ascending=False)` |
| Log 轉換 | `np.log1p(df['col'])` |
| 堆疊長條圖 | `compare.plot(kind='bar', stacked=True)` |
| 群組比較 | `sns.barplot(x=feat, y=target, hue=group)` |
| 交叉表 | `pd.crosstab(df['a'], df['b'], normalize='index')` |
| IQR 離群值 | `Q1 - 1.5*IQR ~ Q3 + 1.5*IQR` |
| 截斷離群值 | `np.clip(series, lower, upper)` |

**下節預告 S4**：Feature Engineering Thinking — 從觀察到創造